# Verification Gems — Standalone Analysis Modules

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/16_verification_gems.ipynb)

Director-AI v3.10 includes 8 standalone verification and analysis modules.
All are **stdlib-only** (zero external dependencies) and work without NLI models.

This notebook covers:
1. Numeric verification — arithmetic, dates, probabilities
2. Reasoning chain verification — non-sequiturs, circularity
3. Temporal freshness — staleness risk for date-sensitive claims
4. Cross-model consensus — pairwise factual agreement
5. Conformal prediction — calibrated P(hallucination) intervals
6. Feedback loop detection — EU AI Act Article 15(4)
7. Agentic loop monitor — circular calls, goal drift, budgets
8. Adversarial robustness — 25-pattern self-test

In [ ]:
!pip install -q director-ai

## 1. Numeric Verification

Catches arithmetic errors, impossible dates, and probabilities outside [0, 100%].
Uses regex extraction + simple arithmetic — no ML needed.

**What it checks:**
- Percentage arithmetic: "grew 15% from X to Y" — is the math right?
- Date ordering: birth < death, founding < present
- Probability bounds: no negative or >100%
- Order of magnitude: Earth population, speed of light
- Internal consistency: same total referenced with different values

In [ ]:
from director_ai.core.verification.numeric_verifier import verify_numeric

result = verify_numeric(
    "Revenue grew 50% from $100 to $120. "
    "She was born in 1990 and died in 1980. "
    "There is a 150% probability of success."
)

print(f"Valid: {result.valid}")
print(f"Claims found: {result.claims_found}")
print(f"Errors: {result.error_count}, Warnings: {result.warning_count}")
print()
for issue in result.issues:
    print(f"  [{issue.severity}] {issue.issue_type}: {issue.description}")

In [ ]:
# Clean text passes without issues
clean = verify_numeric("Water boils at 100 degrees Celsius at sea level.")
print(f"Valid: {clean.valid}, Issues: {clean.error_count}")

## 2. Reasoning Chain Verification

Extracts steps from chain-of-thought responses and verifies each follows
from its premises. Uses Jaccard word overlap as a heuristic (pluggable NLI scorer).

**Verdict types:** `supported`, `non_sequitur`, `unsupported_leap`, `circular`

In [ ]:
from director_ai.core.verification.reasoning_verifier import verify_reasoning_chain

# Valid syllogism
result = verify_reasoning_chain(
    "Step 1: All mammals are warm-blooded. "
    "Step 2: Dogs are mammals. "
    "Step 3: Therefore, dogs are warm-blooded."
)
print(f"Chain valid: {result.chain_valid}")
print(f"Steps: {result.steps_found}, Issues: {result.issues_found}")
for v in result.verdicts:
    print(f"  Step {v.step_index}: {v.verdict} ({v.confidence:.2f})")

In [ ]:
# Non-sequitur: conclusion doesn't follow
result = verify_reasoning_chain(
    "Step 1: All birds can fly. "
    "Step 2: Penguins are birds. "
    "Step 3: Therefore, the economy is growing."
)
print(f"Chain valid: {result.chain_valid}")
for v in result.verdicts:
    print(f"  Step {v.step_index}: {v.verdict} ({v.confidence:.2f}) — {v.reason}")

## 3. Temporal Freshness

Flags claims that may rely on stale knowledge — positions, statistics,
records, and "current" references. Returns staleness risk per claim.

**Claim types:** `position`, `statistic`, `record`, `current_reference`

In [ ]:
from director_ai.core.scoring.temporal_freshness import score_temporal_freshness

result = score_temporal_freshness("The CEO of Apple is Tim Cook.")
print(f"Has temporal claims: {result.has_temporal_claims}")
print(f"Overall staleness risk: {result.overall_staleness_risk:.2f}")
for c in result.claims:
    print(f"  [{c.claim_type}] {c.text} (risk: {c.staleness_risk:.2f})")

In [ ]:
# Timeless facts have zero staleness risk
result = score_temporal_freshness("Water is composed of hydrogen and oxygen.")
print(f"Staleness risk: {result.overall_staleness_risk:.2f}")

## 4. Cross-Model Consensus

Scores pairwise factual agreement across multiple model responses using
Jaccard word overlap (pluggable NLI scorer for production use).

High disagreement between models → low confidence in any individual answer.

In [ ]:
from director_ai.core.scoring.consensus import ConsensusScorer, ModelResponse

scorer = ConsensusScorer(models=["gpt-4o", "claude", "gemini"])
result = scorer.score_responses(
    [
        ModelResponse(model="gpt-4o", response="Paris is the capital of France"),
        ModelResponse(model="claude", response="Paris is the capital of France"),
        ModelResponse(model="gemini", response="Berlin is the capital of Germany"),
    ]
)

print(f"Agreement score: {result.agreement_score:.2f}")
print(f"Has consensus: {result.has_consensus}")
print(f"Models: {result.num_models}")
print()
for p in result.pairs:
    status = "agree" if p.agreed else "DISAGREE"
    print(f"  {p.model_a} vs {p.model_b}: {status} (divergence={p.divergence:.2f})")

## 5. Conformal Prediction Intervals

Calibrated, distribution-free uncertainty on hallucination probability.
Instead of binary approved/rejected, returns a prediction interval:
"95% confident P(hallucination) is between 5% and 15%."

Based on Mohri & Hashimoto, "Conformal Factuality" (ICML 2024).

**Theory:** Split conformal prediction uses nonconformity scores
derived from (guardrail_score, human_label) pairs. The quantile at
level ⌈(n+1)·α⌉/n gives the interval half-width, where α is coverage.

In [ ]:
from director_ai.core.calibration.conformal import ConformalPredictor

predictor = ConformalPredictor(coverage=0.95)

# Calibration data: (score, was_hallucination) pairs
scores = [0.9, 0.85, 0.1, 0.15, 0.88, 0.12] * 6  # 36 samples
labels = [False, False, True, True, False, True] * 6
predictor.calibrate(scores, labels)

# Predict for a new score
for test_score in [0.9, 0.7, 0.3, 0.1]:
    interval = predictor.predict(score=test_score)
    print(
        f"  score={test_score:.1f} → "
        f"P(hallucination) in [{interval.lower:.2f}, {interval.upper:.2f}] "
        f"(point={interval.point_estimate:.2f}, reliable={interval.is_reliable})"
    )

In [ ]:
# Uncalibrated predictor returns full [0, 1] interval
uncal = ConformalPredictor(coverage=0.95)
interval = uncal.predict(score=0.7)
print(
    f"Uncalibrated: [{interval.lower:.2f}, {interval.upper:.2f}], reliable={interval.is_reliable}"
)

## 6. Feedback Loop Detection

EU AI Act Article 15(4) requires: *"eliminate or reduce the risk of
possibly biased outputs influencing input for future operations
(feedback loops)."*

Detects when AI outputs reappear as inputs using trigram Jaccard similarity.
Severity levels: `low`, `medium`, `high`.

In [ ]:
from director_ai.compliance.feedback_loop_detector import FeedbackLoopDetector

detector = FeedbackLoopDetector(similarity_threshold=0.5)

# Record a previous AI output
output = "Machine learning enables systems to learn from data patterns."
detector.record_output(output, timestamp=1.0)

# Check if the same text comes back as input
alert = detector.check_input(output)
if alert:
    print(
        f"Loop detected: similarity={alert.similarity:.2f}, severity={alert.severity}"
    )

# Different input — no loop
alert2 = detector.check_input("What is the weather today?")
print(f"Different input: loop_detected={alert2 is not None}")

## 7. Agentic Loop Monitor

Monitors AI agent execution loops for safety issues:
- **Circular tool calls** — same tool + same args repeated N times
- **Goal drift** — current action diverges from original objective
- **Budget exhaustion** — tokens, steps, or wall time exceeded

Returns per-step verdicts with halt/warn decisions.
The first guardrail product that monitors agent loops.

In [ ]:
from director_ai.agentic.loop_monitor import LoopMonitor

monitor = LoopMonitor(
    goal="Find quarterly revenue for Q3 2025",
    max_steps=10,
    circular_threshold=3,
)

# Simulate an agent loop that gets stuck
actions = [
    ("search_database", "revenue Q3 2025"),
    ("search_database", "revenue Q3 2025"),  # repeat
    ("search_database", "revenue Q3 2025"),  # circular!
    ("format_report", "revenue data"),  # back on track
]

for action, args in actions:
    verdict = monitor.check_step(action=action, args=args, tokens=500)
    print(
        f"  Step {verdict.step_number}: "
        f"halt={verdict.should_halt}, warn={verdict.should_warn}, "
        f"drift={verdict.goal_drift_score:.2f}"
    )
    if verdict.reasons:
        for r in verdict.reasons:
            print(f"    -> {r}")
    if verdict.should_halt:
        break

status = monitor.status()
print(f"\nTotal steps: {status.total_steps}")
print(f"Circular detections: {status.circular_detections}")

## 8. REST API

All gems are exposed via FastAPI endpoints. Start the server:

```bash
director-ai serve --port 8080
```

Then call any gem endpoint:

In [ ]:
# REST endpoint reference (run these with the server running)
endpoints = {
    "POST /v1/verify/numeric": '{"text": "Revenue grew 50% from $100 to $120."}',
    "POST /v1/verify/reasoning": '{"text": "Step 1: A. Step 2: Therefore B."}',
    "POST /v1/temporal-freshness": '{"text": "The CEO of Apple is Tim Cook."}',
    "POST /v1/consensus": '{"responses": [{"model": "a", "response": "..."}, {"model": "b", "response": "..."}]}',
    "POST /v1/conformal/predict": '{"score": 0.7, "calibration_scores": [...], "calibration_labels": [...]}',
    "POST /v1/compliance/feedback-loops": '{"input_text": "...", "previous_outputs": ["..."]}',
    "POST /v1/agentic/check-step": '{"goal": "...", "action": "...", "args": "..."}',
    "POST /v1/adversarial/test": '{"prompt": "...", "response": "..."}',
}

print("Available gem endpoints:\n")
for endpoint, payload in endpoints.items():
    print(f"  {endpoint}")
    print(f"    Body: {payload}\n")

## CLI Quick Reference

```bash
director-ai verify-numeric "Revenue grew 50% from $100 to $120"
director-ai verify-reasoning "Step 1: A. Step 2: Therefore B."
director-ai temporal-freshness "The CEO of Apple is Tim Cook"
director-ai check-step "Find revenue" "search" "revenue Q3"
director-ai consensus "gpt:Paris is the capital" "claude:Paris is the capital"
director-ai adversarial-test
```

## Next Steps

- [Verification Gems Guide](https://anulum.github.io/director-ai/guide/verification-gems/) — full parameter reference
- [API Reference](https://anulum.github.io/director-ai/api/) — all public classes and functions
- [Compliance Reporting](https://anulum.github.io/director-ai/guide/compliance-reporting/) — EU AI Act Article 15
- [Online Calibration](https://anulum.github.io/director-ai/guide/online-calibration/) — threshold tuning from human feedback